# FormulaLM — PROMPT 7: Base Model Baseline Evaluation (Qwen3-1.7B, Non-Thinking)

Evaluates the **untouched** `Qwen/Qwen3-1.7B` base model (pinned revision `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e`, non-thinking mode, `temperature=0`, `max_tokens=96`, `seed=42`) in 4 baseline settings:
1. `baseline_zeroshot_synthetic` — 0-shot on `data/processed/test_synthetic.jsonl` (500 examples)
2. `baseline_5shot_synthetic` — 5-shot (drawn from `train.jsonl`) on `test_synthetic.jsonl` (500 examples)
3. `baseline_zeroshot_real` — 0-shot on `data/test_real/test_real_DRAFT.csv` (80 examples)
4. `baseline_5shot_real` — 5-shot (drawn from `train.jsonl`) on `data/test_real/test_real_DRAFT.csv` (80 examples)

**Hardware:** Colab / Kaggle T4 GPU (16 GB VRAM). Runtime: ~3–5 minutes total with `batch_size=8`.
**Resumable:** every prediction is flushed immediately to `results/baseline_*/predictions.jsonl`.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone the repo (set GITHUB_TOKEN in Colab Secrets if the repo is still private)
import os, pathlib

REPO_DIR = pathlib.Path('/content/FormulaLM')
if not REPO_DIR.exists():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = os.environ.get('GITHUB_TOKEN')
    if token:
        url = f'https://{token}@github.com/abubakarp789/FormulaLM.git'
    else:
        url = 'https://github.com/abubakarp789/FormulaLM.git'
    !git clone {url} /content/FormulaLM
else:
    !git -C /content/FormulaLM pull --ff-only

%cd /content/FormulaLM

In [ ]:
# 3. Install lightweight eval + transformers dependencies
%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate "transformers>=4.51.0" "huggingface_hub>=0.30.0"
%pip install -q --no-deps -e .

In [ ]:
# 4. Regenerate verified train/val/test_synthetic JSONL splits (exact seed=42)
!python -m src.data.build_dataset --workers 2

In [ ]:
# 5. Load Qwen/Qwen3-1.7B ONCE on GPU and run all 4 baseline evaluations
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
SEED = 42
MAX_TOKENS = 96
BATCH_SIZE = 8

backend = TransformersBackend(
    MODEL_ID,
    revision=REVISION,
    load_in_4bit=False,
    enable_thinking=False,
)

runs = [
    ('baseline_zeroshot_real', 'data/test_real/test_real_DRAFT.csv', 0),
    ('baseline_5shot_real', 'data/test_real/test_real_DRAFT.csv', 5),
    ('baseline_zeroshot_synthetic', 'data/processed/test_synthetic.jsonl', 0),
    ('baseline_5shot_synthetic', 'data/processed/test_synthetic.jsonl', 5),
]

for run_name, dataset, few_shot in runs:
    print(f'=== Running {run_name} (few_shot={few_shot}) ===')
    m = evaluate(
        backend=backend,
        dataset_path=dataset,
        run_name=run_name,
        seed=SEED,
        max_tokens=MAX_TOKENS,
        few_shot=few_shot,
        few_shot_source='data/processed/train.jsonl',
        batch_size=BATCH_SIZE,
        resume=True,
    )
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | "
        f"exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | "
        f"avg_lat={ov['avg_latency_ms']:.1f}ms"
    )

In [ ]:
# 6. Produce baseline comparison table and top 15 failure patterns
!python -m src.eval.analyze_errors \
  --runs results/baseline_zeroshot_synthetic results/baseline_5shot_synthetic results/baseline_zeroshot_real results/baseline_5shot_real \
  --out results/baseline_summary.md \
  --json-out results/baseline_summary.json \
  --top-k 15

In [ ]:
# 7. Package results/baseline_* for download back to your local repo
!zip -rq baseline_results.zip results/baseline_*
!ls -lh baseline_results.zip
try:
    from google.colab import files
    files.download('baseline_results.zip')
except Exception:
    print('Download baseline_results.zip from the notebook file browser.')